# Aplicación de transcripción de audio con OpenAI Whisper

**Duración estimada:** 25 minutos

## Propósito
En este cuaderno vas a construir una interfaz gráfica sencilla que permita cargar archivos de audio y obtener su transcripción automática. Para esto, vas a integrar dos herramientas clave:
- **OpenAI Whisper**: un modelo de reconocimiento de voz (ASR) robusto y de código abierto.
- **Gradio**: una librería que permite crear interfaces de usuario para modelos de Machine Learning con muy pocas líneas de código.

Al finalizar, vas a tener una aplicación funcional que puede procesar audio en diversos idiomas y formatos, lista para usar en un entorno de laboratorio o investigación.

## Resultados de aprendizaje
Al terminar este cuaderno vas a poder:
- configurar un pipeline de reconocimiento de voz usando la librería `transformers`;
- definir funciones de procesamiento de audio en Python;
- construir y lanzar una interfaz web interactiva con Gradio.

---
## 1. Instalación de dependencias

Para que la aplicación funcione, necesitás instalar `gradio` (para la interfaz) y `transformers` junto con `torch` (para ejecutar el modelo Whisper).

In [1]:
# Instalamos las librerías necesarias en modo silencioso
!pip install gradio transformers torch accelerate -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 2. Importación de librerías

Ahora cargamos los componentes necesarios. Vamos a usar el `pipeline` de Transformers, que simplifica enormemente la carga y el uso de modelos pre-entrenados.

In [2]:
import torch
from transformers import pipeline
import gradio as gr

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


---
## 3. Configuración de la lógica de transcripción

Vas a definir una función que reciba la ruta de un archivo de audio y devuelva el texto transcrito. El modelo elegido es `whisper-small`, que ofrece un buen equilibrio entre precisión y velocidad de ejecución.

In [3]:
def transcribir_audio(archivo_audio):
    """
    Recibe un archivo de audio y devuelve su transcripción usando Whisper.
    """
    # Inicializamos el pipeline de reconocimiento de voz (ASR)
    # El parámetro chunk_length_s permite procesar audios largos dividiéndolos en fragmentos
    pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small",
        chunk_length_s=30,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    # Procesamos el archivo y extraemos el texto
    resultado = pipe(archivo_audio, batch_size=8)["text"]
    return resultado

---
## 4. Construcción de la interfaz gráfica

Con Gradio, definís los componentes de entrada (`gr.Audio`) y salida (`gr.Textbox`). La clase `gr.Interface` se encarga de conectar esos componentes con la función que definiste anteriormente.

In [4]:
# Definimos el componente de entrada: audio cargado desde la computadora
entrada_audio = gr.Audio(sources="upload", type="filepath", label="Cargar audio")

# Definimos el componente de salida: cuadro de texto para la transcripción
salida_texto = gr.Textbox(label="Transcripción resultante")

# Creamos la interfaz conectando lógica y componentes
demo = gr.Interface(
    fn=transcribir_audio,
    inputs=entrada_audio,
    outputs=salida_texto,
    title="Laboratorio NLP: Transcriptor con Whisper",
    description="Subí un archivo de audio para que el modelo identifique el idioma y genere la transcripción."
)

---
## 5. Lanzamiento de la aplicación

Al ejecutar la siguiente celda, vas a ver la interfaz directamente en el notebook. Si estás en Google Colab, se va a generar un enlace público temporal para que puedas probarla desde otros dispositivos.

In [ ]:
# Lanzamos la aplicación
demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A 

---
## Análisis y reflexión

Considerá los siguientes puntos sobre el funcionamiento de la app:
- **Latencia**: ¿Cuánto tarda el modelo en procesar un minuto de audio? ¿Cómo afecta el uso de GPU o CPU?

Al correr el modelo sin aceleración por hardware, el procesamiento de un audio de aproximadamente 15 minutos tomó un tiempo considerablemente mayor a la duración real del audio. En contraste, con GPU el mismo proceso tomaría aproximadamente 20-30 segundos, siendo hasta 30 veces más rápido. Esto demuestra que sin GPU la transcripción en tiempo real es inviable, siendo útil solo para uso offline o experimental.

- **Precisión**: ¿Cómo maneja el modelo los modismos locales o el ruido de fondo?

El modelo presentó un error de alucinación por repetición, generando frases en bucle en segmentos largos del audio. Esto es un problema conocido de Whisper en audios extensos sin aceleración de hardware. Además, nombres propios poco comunes como 'Tycho Brahe' fueron mal transcritos. Esto muestra que la precisión baja notablemente con audios largos procesados sin GPU.

- **Escalabilidad**: ¿Qué cambios harías para que esta app soporte audios de varias horas de duración?

Para soportar audios de varias horas sería necesario dividir el audio en fragmentos (chunks) y procesarlos secuencialmente para no saturar la memoria, incorporar GPU para reducir drásticamente el tiempo de procesamiento, agregar una barra de progreso que muestre el avance en tiempo real, y guardar resultados parciales para no perder la transcripción ante un error. Estas mejoras harían viable el uso de la app en contextos de producción real.

Este cuaderno demostró que podés pasar de un modelo complejo (Whisper) a una herramienta interactiva en cuestión de minutos. Esta capacidad es fundamental para prototipar soluciones de PLN de manera ágil.